In [0]:
-- Count total deliveries in each match
SELECT match_id, COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY match_id;

-- Find total runs scored by each batting_team
SELECT batting_team, SUM(total_runs) AS total_runs_scored
FROM deliveries
GROUP BY batting_team;

-- Count total wickets taken by each bowler
SELECT bowler, COUNT(*) AS total_wickets
FROM deliveries
WHERE dismissal_type IS NOT NULL
GROUP BY bowler;

-- Find number of matches played in each season
SELECT season, COUNT(*) AS matches_played
FROM matches
GROUP BY season;

-- Get total number of players by nationality
SELECT nationality, COUNT(*) AS total_players
FROM players
GROUP BY nationality;

-- Find top 5 batsmen with highest total runs
SELECT striker AS batsman, SUM(batsman_runs) AS total_runs
FROM deliveries
GROUP BY striker
ORDER BY total_runs DESC
LIMIT 5;

-- Calculate strike rate of each batsman
SELECT striker AS batsman,
       SUM(batsman_runs) AS total_runs,
       COUNT(*) AS balls_faced,
       ROUND((SUM(batsman_runs) * 100.0) / COUNT(*), 2) AS strike_rate
FROM deliveries
GROUP BY striker;

-- Find bowlers with highest number of wickets
SELECT bowler, COUNT(*) AS total_wickets
FROM deliveries
WHERE is_wicket = true AND dismissal_type NOT IN ('run out', 'retired hurt', 'obstructing the field')
GROUP BY bowler
ORDER BY total_wickets DESC
LIMIT 5;

-- Get matches where first innings score > 180
SELECT match_id, season, first_innings_score
FROM matches
WHERE first_innings_score > 180;

-- Find player_of_match count for each player
SELECT player_of_match, COUNT(*) AS awards
FROM matches
GROUP BY player_of_match
ORDER BY awards DESC;

-- Total runs per season by joining deliveries and matches
SELECT m.season, SUM(d.total_runs) AS total_runs
FROM deliveries d
JOIN matches m ON d.match_id = m.match_id
GROUP BY m.season;

-- Top batsman in each season
SELECT season, batsman, total_runs
FROM (
  SELECT m.season, d.striker AS batsman, SUM(d.batsman_runs) AS total_runs,
         ROW_NUMBER() OVER (PARTITION BY m.season ORDER BY SUM(d.batsman_runs) DESC) AS rn
  FROM deliveries d
  JOIN matches m ON d.match_id = m.match_id
  GROUP BY m.season, d.striker
) t
WHERE rn = 1;

-- Winning team with total runs scored in that match
SELECT m.match_id, m.season, m.winner, SUM(d.total_runs) AS total_runs_scored
FROM matches m
JOIN deliveries d ON m.match_id = d.match_id AND d.batting_team = m.winner
GROUP BY m.match_id, m.season, m.winner;

-- City-wise average first innings score
SELECT city, ROUND(AVG(first_innings_score), 2) AS avg_first_innings_score
FROM matches
GROUP BY city;

-- List of players who never got player_of_match
SELECT player_name
FROM players
WHERE player_name NOT IN (
  SELECT DISTINCT player_of_match
  FROM matches
  WHERE player_of_match IS NOT NULL
);

-- Find batsmen with average runs per match > 30
SELECT striker AS batsman,
       COUNT(DISTINCT match_id) AS matches_played,
       SUM(batsman_runs) AS total_runs,
       ROUND(SUM(batsman_runs) / COUNT(DISTINCT match_id), 2) AS avg_runs_per_match
FROM deliveries
GROUP BY striker
HAVING avg_runs_per_match > 30;

-- Calculate economy rate of each bowler
SELECT bowler,
       SUM(total_runs) AS runs_conceded,
       COUNT(*) AS balls_bowled,
       ROUND((SUM(total_runs) * 6.0) / COUNT(*), 2) AS economy_rate
FROM deliveries
GROUP BY bowler;

-- Find matches where chasing team won
SELECT match_id, season, team2 AS chasing_team, winner
FROM matches
WHERE winner = team2;

-- Identify players who scored 50+ runs in a match
SELECT match_id, striker AS batsman, SUM(batsman_runs) AS runs_scored
FROM deliveries
GROUP BY match_id, striker
HAVING SUM(batsman_runs) >= 50;

-- Find highest partnership (striker + non_striker)
SELECT match_id, innings, over, striker, non_striker, SUM(total_runs) AS partnership_runs
FROM deliveries
GROUP BY match_id, innings, over, striker, non_striker
ORDER BY partnership_runs DESC
LIMIT 1;

-- Rank batsmen by total runs within each season
SELECT season, batsman, total_runs,
       RANK() OVER (PARTITION BY season ORDER BY total_runs DESC) AS season_rank
FROM (
  SELECT m.season, d.striker AS batsman, SUM(d.batsman_runs) AS total_runs
  FROM deliveries d
  JOIN matches m ON d.match_id = m.match_id
  GROUP BY m.season, d.striker
) t;

-- Find cumulative runs scored by each batsman over matches
SELECT striker AS batsman, match_id, SUM(batsman_runs) AS runs_in_match,
       SUM(SUM(batsman_runs)) OVER (PARTITION BY striker ORDER BY match_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_runs
FROM deliveries
GROUP BY striker, match_id
ORDER BY striker, match_id;

-- Get top scorer per match using window function
SELECT match_id, batsman, runs_scored
FROM (
  SELECT d.match_id, d.striker AS batsman, SUM(d.batsman_runs) AS runs_scored,
         ROW_NUMBER() OVER (PARTITION BY d.match_id ORDER BY SUM(d.batsman_runs) DESC) AS rn
  FROM deliveries d
  GROUP BY d.match_id, d.striker
) t
WHERE rn = 1;